# nb56 - Physics-locked timing discriminators (H20)

**Error analysis.** nb53: contamination dominates the largest remaining gap. Measured on this data (2026-08-04): the seed cell's signed longitudinal time t_back - t_front is a physics constant +1.23 ns (identical clean/minbias, core width 0.15-0.26 ns, <0.5% violations) - an EM-shower identity card that needs NO window reference time, immune to the in-time-pileup degeneracy that killed H9/H9b.

**Question.** Do per-cell physics-locked timing discriminators - the longitudinal-development pull and a one-sided TOF lateness - reduce contamination-driven error?

**Hypotheses.** H20a `lgpull`: per-cell (t_b - t_f - C_EM)/sigma_diff flags non-EM deposits (KOTO precedent: signed front-back depth timing gives neutron suppression 5.6e5 at 70% eff, arXiv:2411.11237, 2309.12063). H20b `late`: one-sided lateness relu(t - t0_window) tags massive neutrals (TOF delay (L/c)(1/beta-1): K_L +4.9 ns at 1 GeV, n +4.4 ns at 2 GeV at L=12.5 m; CMS one-sided timing veto 0911.4044); differs from falsified H9b by SIGN - only LATE is suspicious. H20c `both`.

**Known ceiling (stated up front).** Both discriminate HADRONIC pileup only; overlapping photons (pi0 daughters from other vertices) share the EM signature. The measured gain bounds the hadronic fraction of our contamination.

**Proof criterion.** EMA recipe (quant + clean-aux + EMA 0.999), 2 seeds per config; anchor 0.0424 +/- 0.0003. Win = >0.002 overall or in any E>17 bin; diagnostic = high-contamination-tertile width per config. C_EM and sigma_t(E) measured from clean in this notebook - no free parameters.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from scipy.optimize import curve_fit
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
from picocal_data import build_grid, splits_for, THRESH
from picocal_models import QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB56_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB56_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
print(f'device {DEVICE} | mode {MODE} | build {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events
device cuda | mode full | build 146s


In [2]:
W = 4
pairs = []; seed_dt = []
for ev in CE:
    m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
    tf, tb, e, fr, bk = ev['tf'][m], ev['tb'][m], ev['e'][m], ev['fr'][m], ev['bk'][m]
    for t, en in ((tf, fr), (tb, bk)):
        v = np.isfinite(t)
        if v.sum() >= 3:
            pairs.append(np.stack([en[v], t[v] - np.median(t[v])], 1))
    s = int(np.argmax(e))
    if np.isfinite(tf[s]) and np.isfinite(tb[s]):
        seed_dt.append(tb[s] - tf[s])
pairs = np.concatenate(pairs); pairs = pairs[pairs[:, 0] > 0]
def rsig(x):
    x = np.sort(np.asarray(x)); n = len(x); k = max(1, int(0.683 * n))
    return 0.5 * np.min(x[k:] - x[:n-k])
edges = np.quantile(pairs[:, 0], np.linspace(0, 1, 13))
ec, sv = [], []
for i in range(12):
    hi = edges[i+1] + (1e-9 if i == 11 else 0)
    mm = (pairs[:, 0] >= edges[i]) & (pairs[:, 0] < hi)
    s = rsig(pairs[mm, 1] - np.median(pairs[mm, 1]))
    if np.isfinite(s): ec.append(float(np.median(pairs[mm, 0]))); sv.append(s)
(A_T, B_T), _ = curve_fit(lambda E, A, B: np.sqrt((A/E)**2 + B**2), np.array(ec), np.array(sv), p0=[1.5, 0.25], maxfev=10000)
A_T, B_T = float(abs(A_T)), float(abs(B_T))
def sigt(E): return np.sqrt((A_T / np.maximum(E, 1e-3)) ** 2 + B_T ** 2)
C_EM = float(np.median(seed_dt))
print(f'sigma_t(E) = ({A_T:.2f}/E) (+) {B_T:.3f} ns | C_EM = {C_EM:+.3f} ns [{len(seed_dt)} clean seeds]')

sigma_t(E) = (1.41/E) (+) 0.245 ns | C_EM = +1.233 ns [30033 clean seeds]


In [3]:
NC = 11; NG = 6
def make_windows_phys(EVS, use_lg, use_late):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        dtf = np.where(np.isfinite(tf), tf - t0f, np.nan)
        dtb = np.where(np.isfinite(tb), tb - t0b, np.nan)
        tfc = np.where(np.isfinite(dtf), np.clip(dtf, -5, 5), 0.0); htf = np.isfinite(dtf).astype(np.float32)
        tbc = np.where(np.isfinite(dtb), np.clip(dtb, -5, 5), 0.0); htb = np.isfinite(dtb).astype(np.float32)
        vb = np.isfinite(tf) & np.isfinite(tb)
        sd = np.sqrt(sigt(np.maximum(fr, 1e-3)) ** 2 + sigt(np.maximum(bk, 1e-3)) ** 2)
        lg = np.where(vb & (use_lg > 0), np.clip((tb - tf - C_EM) / sd, -5, 5), 0.0)
        latemin = np.nanmin(np.stack([dtf, dtb]), 0)
        late = np.where(np.isfinite(latemin) & (use_late > 0), np.clip(np.maximum(latemin, 0.0), 0, 10), 0.0)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), tfc, tbc,
                         lg.astype(np.float32), late.astype(np.float32)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'], ev['reg']))
        keep.append(i)
    return rows, np.array(keep)
def prep_phys(use_lg, use_late):
    rows, keep = make_windows_phys(ME, use_lg, use_late)
    ktr, kva, kte = splits_for(keep, len(ME))
    crows, _ = make_windows_phys(CE, use_lg, use_late)
    n_mb = len(rows); rows = rows + crows
    ctr = np.arange(n_mb, len(rows))
    N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
    y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
    Et = np.array([r[3] for r in rows], np.float32)
    sumE = np.array([r[1] for r in rows], np.float32)
    X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
    G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
    for i, (tok, se, sde, et, rg) in enumerate(rows):
        n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
        e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat, float(i >= n_mb)]
    la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
    G[:, :5] = (G[:, :5] - G[ktr, :5].mean(0)) / (G[ktr, :5].std(0) + EPS)
    cont2 = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
    mean = cont2.mean(0); std = cont2.std(0) + EPS
    X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
    T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
             G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
             E=torch.from_numpy(Eraw).to(DEVICE))
    return dict(T=T, y=y, Et=Et, sumE=sumE, ktr=ktr, kva=kva, kte=kte, ctr=ctr,
                IN_DIM=IN_DIM, la0=float(la0), lb0=float(lb0))

In [4]:
import torch.nn as nn
class SubNetP(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = torch.sigmoid(self.fhead(h).squeeze(-1)) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
QS = torch.tensor(QUANTILES, device=DEVICE)
def train_eval(P, config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetP(P['IN_DIM'], P['la0'], P['lb0']).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    T = P['T']
    tr_idx = np.concatenate([np.asarray(P['ktr']), P['ctr']])
    ck = CKPT / f'nb56_{config}_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m_, b): return m_(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
    def vloss(m_):
        m_.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(P['kva'], 256):
                d = T['Y'][b] - fwd(m_, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from ep {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            d = T['Y'][b] - fwd(model, b)
            torch.maximum(QS * d, (QS - 1) * d).mean().backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetP(P['IN_DIM'], P['la0'], P['lb0']).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(P['kva']), run(P['kte']), P['y'][P['kva']])
    return float(resolution(pe, P['Et'][P['kte']])['sigma_eff']), pe

In [5]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
CONFS = dict(lgpull=(1, 0), late=(0, 1), both=(1, 1))
JOBS = {'smoke': [('lgpull', 0), ('late', 0)],
        'full': [(cfg, s) for cfg in ('lgpull', 'late', 'both') for s in (0, 1)]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb56_physics{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
PREPS = {}
for config, seed in JOBS:
    if (config, seed) in done: print('skip', config, seed); continue
    if config not in PREPS: PREPS[config] = prep_phys(*CONFS[config])
    t1 = time.time()
    sig, pe = train_eval(PREPS[config], config, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb56_pred{TAG}_{config}_s{seed}.npy', pe)
    row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

/tmp/ipykernel_1625219/3150719152.py:17: RuntimeWarning: All-NaN slice encountered
  latemin = np.nanmin(np.stack([dtf, dtb]), 0)


lgpull seed 0: sigma_eff 0.0425 (1081s)


lgpull seed 1: sigma_eff 0.0419 (1365s)


/tmp/ipykernel_1625219/3150719152.py:17: RuntimeWarning: All-NaN slice encountered
  latemin = np.nanmin(np.stack([dtf, dtb]), 0)


late seed 0: sigma_eff 0.0432 (1364s)


late seed 1: sigma_eff 0.0426 (1300s)


/tmp/ipykernel_1625219/3150719152.py:17: RuntimeWarning: All-NaN slice encountered
  latemin = np.nanmin(np.stack([dtf, dtb]), 0)


both seed 0: sigma_eff 0.0424 (1145s)


both seed 1: sigma_eff 0.0424 (1255s)


config  seed  sigma_eff  elapsed
lgpull     0     0.0425     1081
lgpull     1     0.0419     1365
  late     0     0.0432     1364
  late     1     0.0426     1300
  both     0     0.0424     1145
  both     1     0.0424     1255


## Verdict

Anchor 0.0424 +/- 0.0003 (EMA recipe, same everything minus the physics columns). Win = >0.002 overall or any E>17 bin. Diagnostic: width in contamination tertiles - physics discriminators must help most in the high-contamination third; the gain size bounds the hadronic fraction of our pileup.

In [6]:
print('anchor: EMA singles 0.0424 +/- 0.0003 | stack record 0.0409 (LS-calib 0.0415)')
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
for cfg in ('lgpull', 'late', 'both'):
    if cfg not in PREPS: continue
    P = PREPS[cfg]
    te_e = P['Et'][P['kte']]
    preds = [np.load(OUT / f'nb56_pred{TAG}_{cfg}_s{s}.npy') for s in SEEDS
             if (OUT / f'nb56_pred{TAG}_{cfg}_s{s}.npy').exists()]
    if not preds: continue
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        bins.append(f'{resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}')
    cont = P['sumE'][P['kte']] / np.maximum(1000.0 * te_e, EPS)
    qs = np.quantile(cont, [1/3, 2/3])
    terts = []
    for gsel, lab in [(cont < qs[0], 'loC'), ((cont >= qs[0]) & (cont < qs[1]), 'miC'), (cont >= qs[1], 'hiC')]:
        terts.append(f'{lab}:{resolution(ens[gsel], te_e[gsel])["sigma_eff"]:.4f}')
    print(f'{cfg:6s} mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}')
    print('       per-bin ' + ' / '.join(bins) + ' | ' + ' '.join(terts))

anchor: EMA singles 0.0424 +/- 0.0003 | stack record 0.0409 (LS-calib 0.0415)
lgpull mean 0.0422 +/- 0.0003 | ens 0.0417
       per-bin 0.0648 / 0.0469 / 0.0356 / 0.0354 / 0.0332 / 0.0363 | loC:0.0280 miC:0.0359 hiC:0.0731
late   mean 0.0429 +/- 0.0003 | ens 0.0425
       per-bin 0.0658 / 0.0479 / 0.0360 / 0.0343 / 0.0341 / 0.0364 | loC:0.0279 miC:0.0361 hiC:0.0742
both   mean 0.0424 +/- 0.0000 | ens 0.0418
       per-bin 0.0643 / 0.0466 / 0.0361 / 0.0348 / 0.0341 / 0.0366 | loC:0.0279 miC:0.0363 hiC:0.0747
